# AlphaCluster — Simple Actions Diagnostic (3-Action Mode)

Diagnostic training run on Google Colab GPU to test whether a simplified 3-action space
(long / flat / short) helps the agent learn faster than the full 36-action space.

**Note:** The notebook is edited locally in VS Code but the kernel runs on Colab's remote runtime.
Source code and data are read from Google Drive.

## What this tests

- **CNN+Transformer** feature extractor (replaces pure CNN)
- **19-feature market observations** (OHLCV + 14 technical indicators)
- **12-feature account state** (7 original + 5 trade-tracking features)
- **8-component reward** (asymmetric PnL, fee penalty, opportunity cost, position management with sqrt ramp, trade completion, churn penalty, drawdown, trade quality bonus)
- **Curriculum learning** (3 phases: learn to trade (0-30%) → learn quality (30-60%) → refine & exploit (60-100%))
- **Parallel environments** (4x SubprocVecEnv + reward normalization)
- **3-action space** — long / flat / short with fixed 10% size and 10x leverage
- **Isolated margin** — liquidation loses only the allocated margin, not the entire account; episode continues

## Why 500k steps?

Fewer actions means faster convergence is expected. 500k steps is sufficient to diagnose
whether the agent can learn a directional edge before committing to a full 2M-step run.

## Setup

Upload to Google Drive (`My Drive/alphacluster/`):
1. `src/` — the project source directory (copy from your local repo)
2. `data/btcusdt_5m.parquet` — candle data
3. `data/btcusdt_funding.parquet` — funding rates (optional)

Trained models will be saved back to `My Drive/alphacluster/models/`.

## Cell 1 — Mount Drive & Install Dependencies

In [1]:
import os
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/alphacluster")
DRIVE_SRC = DRIVE_ROOT / "src"

assert DRIVE_SRC.exists(), (
    f"Source not found at {DRIVE_SRC}\n"
    f"Copy your local src/ directory to Google Drive: My Drive/alphacluster/src/"
)

# Copy source to local runtime storage to avoid Drive disconnection issues
LOCAL_SRC = Path("/content/src")
if LOCAL_SRC.exists():
    shutil.rmtree(LOCAL_SRC)
shutil.copytree(DRIVE_SRC, LOCAL_SRC)
print(f"Copied source to {LOCAL_SRC}")

# Install dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "stable-baselines3>=2.4,<3.0", "gymnasium>=1.0,<2.0",
    "pyarrow", "python-dotenv", "tqdm", "rich"], check=True)

# Add local copy to path
PROJECT_ROOT = DRIVE_ROOT
sys.path.insert(0, str(LOCAL_SRC))

import alphacluster
print(f"AlphaCluster loaded from {Path(alphacluster.__file__).parent}")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Enable GPU runtime: Runtime > Change runtime type > T4 GPU")

## Cell 2 — Load & Split Data

In [2]:
import pandas as pd

DATA_DIR = Path("/content/drive/MyDrive/alphacluster/data")
KLINES_PATH = DATA_DIR / "btcusdt_5m.parquet"
FUNDING_PATH = DATA_DIR / "btcusdt_funding.parquet"

assert KLINES_PATH.exists(), (
    f"Kline data not found at {KLINES_PATH}\n"
    f"Upload btcusdt_5m.parquet to Google Drive: My Drive/alphacluster/data/"
)

klines_df = pd.read_parquet(KLINES_PATH, engine="pyarrow")
print(f"Loaded {len(klines_df):,} candles")
print(f"Date range: {klines_df.iloc[0, 0]} to {klines_df.iloc[-1, 0]}")

funding_df = None
if FUNDING_PATH.exists():
    funding_df = pd.read_parquet(FUNDING_PATH, engine="pyarrow")
    print(f"Loaded {len(funding_df):,} funding records")
else:
    print("No funding data found; training without funding rates.")

# Chronological split: 70% train / 15% val / 15% test
n = len(klines_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = klines_df.iloc[:train_end].reset_index(drop=True)
val_df = klines_df.iloc[train_end:val_end].reset_index(drop=True)
test_df = klines_df.iloc[val_end:].reset_index(drop=True)

print(f"\nData split: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

## Cell 3 — Create Environments

In [3]:
from alphacluster.agent.config import TrainingConfig
from alphacluster.env.trading_env import TradingEnv
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize

config = TrainingConfig(
    total_timesteps=500_000,
    simple_actions=True,
    fixed_size_pct=0.10,
    fixed_leverage=10,
)
config.eval_freq = 50_000

N_ENVS = config.n_envs  # 4 parallel environments

def make_train_env(rank: int):
    """Factory that returns a closure for SubprocVecEnv."""
    def _init():
        import os
        import warnings

        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")

        env = TradingEnv(
            df=train_df,
            funding_df=funding_df,
            window_size=config.window_size,
            episode_length=config.episode_length,
            simple_actions=config.simple_actions,
            fixed_size_pct=config.fixed_size_pct,
            fixed_leverage=config.fixed_leverage,
        )
        env.reset(seed=rank)
        return env
    return _init

print(f"Creating {N_ENVS} parallel training environments ...")
envs = SubprocVecEnv([make_train_env(i) for i in range(N_ENVS)])
train_env = VecNormalize(envs, norm_obs=False, norm_reward=True, clip_reward=10.0)

eval_env = None
if len(val_df) > config.window_size + config.episode_length:
    print("Creating validation environment ...")
    eval_env = TradingEnv(
        df=val_df,
        funding_df=funding_df,
        window_size=config.window_size,
        episode_length=config.episode_length,
        simple_actions=config.simple_actions,
        fixed_size_pct=config.fixed_size_pct,
        fixed_leverage=config.fixed_leverage,
    )
else:
    print("Validation set too small for evaluation; skipping eval callback.")

print(f"\nConfig: lr={config.learning_rate}, batch={config.batch_size}, "
      f"gamma={config.gamma}, ent_coef={config.ent_coef}")
print(f"Timesteps: {config.total_timesteps:,}")
print(f"Eval/checkpoint freq: {config.eval_freq:,}")
print(f"Parallel envs: {N_ENVS} (with VecNormalize reward normalization)")
print(f"Simple actions: {config.simple_actions} (3-action space: long/flat/short)")
print(f"Fixed size: {config.fixed_size_pct*100:.0f}%  Fixed leverage: {config.fixed_leverage}x")

## Cell 4 — Train

In [4]:
import time
from alphacluster.agent.trainer import create_agent, save_agent, train
from alphacluster.config import MODEL_VERSION
from stable_baselines3.common.callbacks import BaseCallback

# Checkpoints go to local storage (fast); final model goes to Drive (persistent)
LOCAL_CHECKPOINT_DIR = Path("/content/checkpoints_simple")
MODELS_DIR = Path("/content/drive/MyDrive/alphacluster/models")


class ProgressCallback(BaseCallback):
    """Print a one-line progress update every ``log_interval`` timesteps."""

    def __init__(self, total_timesteps: int, log_interval: int = 1_000):
        super().__init__(verbose=0)
        self.total_timesteps = total_timesteps
        self.log_interval = log_interval
        self._start_time = None
        self._next_log = log_interval

    def _on_training_start(self) -> None:
        self._start_time = time.time()

    def _on_step(self) -> bool:
        if self.num_timesteps >= self._next_log:
            elapsed = time.time() - self._start_time
            pct = self.num_timesteps / self.total_timesteps * 100
            fps = self.num_timesteps / max(elapsed, 1)
            eta = (self.total_timesteps - self.num_timesteps) / max(fps, 1)
            print(
                f"  [{pct:5.1f}%] {self.num_timesteps:>10,} / {self.total_timesteps:,} "
                f"| {fps:.0f} fps | ETA {eta/60:.0f}m",
                flush=True,
            )
            self._next_log += self.log_interval
        return True


print("Creating PPO agent (CNN+Transformer) ...")
agent = create_agent(train_env, config, verbose=0)

print(f"Training for {config.total_timesteps:,} timesteps ...")
print(f"Model version: {MODEL_VERSION}")
print(f"Checkpoints (local): {LOCAL_CHECKPOINT_DIR}")
print(f"Final model (Drive): {MODELS_DIR}")
print(f"Curriculum phases: Learn to Trade (0-30%) -> Learn Quality (30-60%) -> Refine & Exploit (60-100%)\n")

progress_cb = ProgressCallback(config.total_timesteps, log_interval=10_000)

agent = train(
    agent=agent,
    config=config,
    eval_env=eval_env,
    checkpoint_dir=str(LOCAL_CHECKPOINT_DIR),
    run_tournament=False,
    progress_bar=False,
    verbose=0,
    extra_callbacks=[progress_cb],
)

# Save final model to Drive
MODELS_DIR.mkdir(parents=True, exist_ok=True)
final_path = MODELS_DIR / "ppo_trading_simple_final"
save_agent(agent, final_path)
print(f"\nFinal model saved to {final_path}.pt")

## Cell 4.5 — Learning Curves

## Cell 5 — Evaluate on Test Set

In [5]:
from alphacluster.backtest.runner import run_backtest
from alphacluster.backtest.metrics import calculate_metrics, print_report

EVAL_SEED = 42
N_EVAL_EPISODES = 5

print(f"Running backtest on test set ({N_EVAL_EPISODES} episodes, seed={EVAL_SEED}) ...")
test_env = TradingEnv(
    df=test_df,
    funding_df=funding_df,
    window_size=config.window_size,
    episode_length=config.episode_length,
    simple_actions=config.simple_actions,
    fixed_size_pct=config.fixed_size_pct,
    fixed_leverage=config.fixed_leverage,
)

result = run_backtest(model=agent, env=test_env, n_episodes=N_EVAL_EPISODES, seed=EVAL_SEED)
metrics = calculate_metrics(result)
print_report(metrics)

## Cell 6 — Summary

In [6]:
import os

print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print()
print(f"Model files on Google Drive: {MODELS_DIR}")
print()

for root, dirs, files in os.walk(str(MODELS_DIR)):
    level = root.replace(str(MODELS_DIR), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024 * 1024)
        print(f"{sub_indent}{f}  ({size_mb:.1f} MB)")

print()
print("Models persist on Drive. To use locally:")
print("  1. Download ppo_trading_final.pt from Drive: alphacluster/models/")
print("  2. Place it in your local models/ directory")
print("  3. Run: python scripts/evaluate.py --model-path models/ppo_trading_final.pt")

In [7]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = LOCAL_CHECKPOINT_DIR / "training_metrics.csv"
if metrics_path.exists():
    df = pd.read_csv(metrics_path)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f"Training Progress — {MODEL_VERSION}", fontsize=14)

    plots = [
        ("timestep", "mean_reward", "Mean Episode Return (%)"),
        ("timestep", "trades_per_episode", "Trades per Episode"),
        ("timestep", "win_rate", "Win Rate (%)"),
        ("timestep", "avg_trade_duration", "Avg Trade Duration (steps)"),
        ("timestep", "flat_pct", "Flat Time (%)"),
        ("timestep", "total_pnl_pct", "Total PnL (%)"),
    ]

    for ax, (x, y, title) in zip(axes.flat, plots):
        ax.plot(df[x], df[y], marker="o", markersize=3)
        ax.set_title(title)
        ax.set_xlabel("Timesteps")
        ax.grid(True, alpha=0.3)

    # Mark curriculum phase boundaries
    for ax in axes.flat:
        for pct, label in [(0.3, "Phase 2"), (0.6, "Phase 3")]:
            ts = int(config.total_timesteps * pct)
            ax.axvline(x=ts, color="red", linestyle="--", alpha=0.5)
            ax.text(ts, ax.get_ylim()[1], f" {label}", color="red", fontsize=8, va="top")

    plt.tight_layout()
    plt.show()

    print(f"\nMetrics logged to: {metrics_path}")
    print(df.to_string(index=False))
else:
    print("No training metrics found. Run training first.")